# Rung 1 — the content-based induction task (issue #200)

One Kaggle/Colab session, decided **before** seeing results. The
pre-registration is `07_capstone/notes/variable-distance-induction-preregistration.md`
(and its 2026-10-22 addendum) and the design is ADR-0033.

On the periodic task the previous occurrence of a token is always at `q-L`,
so "attend to `q-L+1`" and "attend to the token after the previous
occurrence" are the *same function* on the training distribution, and the
2026-10-19 probe showed the model had learned the offset. This task puts
the repeats at *varying* distances, so the fixed-offset shortcut does not
exist: the best single fixed offset explains a small fraction of the copy
queries (measured on the data before this run), while a content-matching
head can reach 100%.

Three things changed since this notebook was first written. All three were
decided **before** the run, and all three are recorded in the
pre-registration's 2026-10-22 addendum:

| Change | Where | Why |
|---|---|---|
| **Vocabulary 2048 → 256** | `--content-standard` | The periodic task's 2048 answered a constraint this task does not have: its prefix is sampled *with replacement* (so a large vocabulary is what keeps the prefix collision-free), while the content task's key is sampled *without* replacement. The standard-geometry CPU probe of 2026-10-22 was still at the uniform prediction after 400 epochs at 2048. |
| **Stop rule `mechanism` → `any`** | Step 1 | The two-step rule waits for a head *above* the first layer, and the periodic circuit was entirely in layer 0 — the rule would never fire on this architecture, and the emergence run would silently consume the whole budget. The question the emergence run asks is "when does a copy head appear?", and `any` answers it. |
| **The verdict is a module, and it runs last** | Step 6 | The verdict used to live in a notebook cell that read `mean_path_patch_effect` out of the **exp1** manifest — a metric only the **exp4** manifest carries. It raised `KeyError` on the first seed, after the session was spent. The criteria now live in `src/experiments/content_verdict.py`, the exp4 manifest is read from where it is written, and the verdict cell is the **last** one: a verdict that crashes must not cost the session. |

| Step | What runs | Why |
|------|-----------|-----|
| 0 | **Pilot**: 20 epochs of the content config, timed | Measures this GPU's real throughput (and the fresh-batch data generation, which is pure Python) so the session budget is planned from a measurement instead of an estimate |
| 1 | `exp1 --content-standard --stop-on-confirmed --stop-rule any` | The **emergence** measurement: when does the copy head appear? Ends each seed at the first confirmation, twice in a row |
| 2 | the same, no early stop | The **full-budget** measurement: 3000 epochs (384k optimizer steps per seed). The model it leaves behind is the one Rungs 4 and 5 analyse |
| 3 | `exp4 --from-checkpoint <exp1's checkpoint dir> --source-task variable --counterfactual minimal-pair` | Activation patching + head ablation + path patching **on the very model Rung 1 trained**, measured at a *copy query* |
| 4 | `exp5 --activations-from <rolling checkpoint> --source-task variable` | The SAE on real activations from the same model |
| 5 | zip + download | From here on the artifacts cannot be lost |
| 6 | the verdict, by code | Seven frozen criteria, joined across the two manifests |

**The verdict is computed by code, not by hand.** A seed is never dropped and
the run is never repeated "to get a better label".


In [ ]:
BRANCH = "main"  # a pushed branch or tag — pin it so the manifest's git_sha is a real commit
!git clone --branch $BRANCH --depth 1 https://github.com/AlessioBrillo/from-gradient-to-transformer.git repo
%cd repo
!git rev-parse --short HEAD


In [ ]:
# The GPU images already ship the CUDA torch stack plus numpy, matplotlib, tqdm and
# pyyaml. `uv sync --frozen` would re-resolve the ~5GB CUDA wheel set into the
# notebook's limited working disk and die mid-download — exactly how the first
# grokking kernel errored (2026-10-14). Use the preinstalled stack and fill only a
# real gap; the manifests record the torch/python versions actually used.
import importlib.util
import sys

missing = [
    m
    for m in ("torch", "numpy", "matplotlib", "tqdm", "yaml")
    if importlib.util.find_spec(m) is None
]
if missing:
    !pip install -q {' '.join(missing)}

import torch

assert torch.cuda.is_available(), "no GPU — enable one before running"
print(torch.__version__, "|", torch.cuda.get_device_name(0), "| py", sys.version.split()[0])


## Step 0 — Pilot: measure this GPU before spending the session

Twenty epochs of the **content config** (same shape, same fresh-batch
regime, same detection cadence), timed. The output is the number the
budget has to be planned from: the old notebook estimated "~2h" per run,
which was never measured.

If the projection does not fit the session, the fix is **one seed per
session** (set `SEEDS` below to a single seed), never a shorter budget: a
truncated run is exactly how this repository produced under-trained false
negatives.


In [ ]:
# 20 epochs of the content config: 20 * 128 = 2560 optimizer steps.
# The flags are explicit (not --content-standard) because the pilot needs its
# own epoch count; everything else is the content config.
import os, time
t0 = time.time()
!python -m src.experiments.exp1_induction_heads --task variable --vocab-size 256 --seq-len 64 --d-model 64 --n-layers 2 --n-heads 4 --epochs 20 --batch-size 64 --num-train 8192 --batches fresh --seeds 0 --detect-every 10 --manifest-path results/probe_pilot_content.json
pilot_seconds = time.time() - t0
steps = 20 * 128
rate = steps / max(pilot_seconds, 1e-6)
full = 3000 * 128
print(f"pilot: {pilot_seconds:.1f}s for {steps} steps -> {rate:.1f} steps/s")
print(f"projected standard run: {full / max(rate, 1e-6) / 60:.1f} min per seed")
print(f"projected 3 seeds:       {3 * full / max(rate, 1e-6) / 3600:.2f} h")


## Step 1 — The emergence measurement

How early does the copy head appear? `--stop-on-confirmed --stop-rule any`
ends each seed as soon as the detector confirms a copy head — at two
consecutive checks.

**Why `any` and not `mechanism`:** the `mechanism` rule waits for a head
*above the first layer*, i.e. for the two-step circuit. The periodic task's
circuit was entirely in layer 0, and the 2026-10-22 CPU de-risk confirmed
no head at all — so the two-step rule had a good chance of never firing and
silently turning the emergence run into a second full-budget run (twice the
GPU). The question this step asks is "when does a copy head appear?", and
`any` answers it. The full run below records the confirmation epoch anyway.

`SEEDS` is an environment variable so a short session can run one seed at
a time; the checkpoints and the per-seed manifest are the same either way.


In [ ]:
import os
os.environ["SEEDS"] = "0,1,2"  # set to a single seed (e.g. "0") if the session is short
!python -m src.experiments.exp1_induction_heads --content-standard --seeds $SEEDS --checkpoint-dir checkpoints/induction_content_emergence --checkpoint-every 250 --resume --detect-every 50 --stop-on-confirmed --stop-rule any --manifest-path results/induction_content_emergence.json


## Step 2 — The full-budget run (the measurement Rungs 4 and 5 use)

The content task's standard config (`--content-standard`: vocab 256, seq 64,
d_model 64, 2 layers, 4 heads, fresh batches, 3000 epochs = 384k optimizer
steps per seed), at its full budget, no early stop. The model this run
leaves behind is the one the circuit-patching and SAE rungs analyse.

Why a second run and not just step 1: an early stop ends the run at the
*first* confirmation, which is a different question from "what does the
model settle into at the full budget".


In [ ]:
!python -m src.experiments.exp1_induction_heads --content-standard --seeds $SEEDS --checkpoint-dir checkpoints/induction_content --checkpoint-every 250 --resume --detect-every 50 --manifest-path results/induction_content.json


## Step 3 — Rung 4 on the very model Rung 1 trained

The checkpoint is loaded, not retrained. That is what makes the cascade one
model instead of two, and `--source-task variable` makes it analyse
variable-distance data and measure the interventions at a copy query.

**The cascade runs when the *behaviour* is content-based**, not only when a
per-head detector fires: a copy spread over several heads is invisible to
the 0.30 screen in every one of them, and the group intervention exists
precisely for that case.

`--counterfactual minimal-pair` builds the corrupted half of the
intervention by changing **one token** of the same sequence — the token at
the source of the copy — instead of taking a different sequence. The
2026-10-20 cascade compared a sequence against an unrelated one, which is
why its activation-patching recovery (0.0834) is hard to read as "this
circuit carries the answer".


In [ ]:
import json

m_full = json.load(open("results/induction_content.json"))
m_emergence = json.load(open("results/induction_content_emergence.json"))

BEHAVIOUR_IS_CONTENT = any(
    float(m["copy_accuracy"]) - float(m["fixed_offset_baseline"]) >= 0.20
    for m in m_full["per_seed_metrics"]
)
FOUND_HEAD = any(
    int(m["total_induction_heads"]) > 0 for m in m_full["per_seed_metrics"]
)
print("content-based behaviour:", BEHAVIOUR_IS_CONTENT, "| confirmed head:", FOUND_HEAD)

if BEHAVIOUR_IS_CONTENT or FOUND_HEAD:
    !python -m src.experiments.exp4_circuit_patching --from-checkpoint checkpoints/induction_content --source-vocab-size 256 --source-seq-len 64 --source-d-model 64 --source-n-layers 2 --source-n-heads 4 --source-task variable --counterfactual minimal-pair --seeds $SEEDS --manifest-path results/exp4_content.json
else:
    print("Neither a confirmed head nor a content-based behaviour: Rung 4 has no")
    print("target and is skipped. The verdict in Step 6 says so explicitly.")


In [ ]:
# The cascade's numbers live in the exp4 manifest — read them from there.
# (The old notebook read them out of the exp1 manifest, which never carried
# them: that is the KeyError the verdict module replaced.)
import json
from pathlib import Path

if Path("results/exp4_content.json").exists():
    m_exp4 = json.load(open("results/exp4_content.json"))
    for seed, metrics in zip(m_exp4["seeds"], m_exp4["per_seed_metrics"]):
        print(f"exp4 seed {seed}: heads={int(metrics['total_induction_heads'])} "
              f"(candidates {int(metrics['n_candidate_heads'])}) | "
              f"ablation {metrics['mean_ablation_effect']:.3f} | "
              f"path patch {metrics['mean_path_patch_effect']:.3f} | "
              f"group path patch {metrics['group_path_patch_effect']:.3f} | "
              f"activation recovery {metrics['mean_activation_recovery']:.3f} | "
              f"checkpoint epoch {int(metrics['checkpoint_epoch'])}")
else:
    print("no exp4 manifest: the cascade did not run")


## Step 4 — Rung 5: the SAE on real activations

The same model, the same data, the same checkpoint: the SAE is trained on
the activations Rung 1's model actually produces, and it is skipped (not
faked) when no seed confirmed a head.


In [ ]:
import glob
import os


def first_head_seed(manifest):
    if not os.path.exists(manifest):
        return None
    m = json.load(open(manifest))
    for seed, metrics in zip(m["seeds"], m["per_seed_metrics"]):
        if int(metrics["total_induction_heads"]) > 0:
            return seed
    return None


seed = first_head_seed("results/induction_content.json")
ckpt_dir = "checkpoints/induction_content"
if seed is None:
    seed = first_head_seed("results/induction_content_emergence.json")
    ckpt_dir = "checkpoints/induction_content_emergence"

if seed is None:
    print("No seed confirmed a head — the exp5 real-activation run has no")
    print("confirmed-head checkpoint to harvest from. It is skipped, not faked.")
else:
    ckpt = glob.glob(f"{ckpt_dir}/*seed{seed}*.pt")
    print("harvesting from", ckpt)
    !python -m src.experiments.exp5_sae_dashboard --activations-from {ckpt[0]} --source-vocab-size 256 --source-seq-len 64 --source-d-model 64 --source-n-layers 2 --source-n-heads 4 --source-task variable --seeds $SEEDS --manifest-path results/exp5_content.json


In [ ]:
# Collect every manifest this session produced, then download.
# This cell runs BEFORE the verdict on purpose: the verdict is the last
# thing that can fail, and it must not be able to cost the session.
import glob
import os

manifests = sorted(
    glob.glob("results/induction_content*.json")
    + glob.glob("results/exp4_content.json")
    + glob.glob("results/exp5_content.json")
)
manifests = [p for p in manifests if "probe_" not in p]
print("manifests:", manifests)
!zip -j induction_content_results.zip $manifests
try:
    from google.colab import files

    files.download("induction_content_results.zip")
except ImportError:
    print("Kaggle: the zip is in the notebook output (/kaggle/working).")


## Step 6 — The verdict, computed from the manifests

The seven pre-registered criteria, in the order they were frozen. A
criterion is met when the *seed* meets it; the run is POSITIVE when a
majority of seeds meet all seven, PARTIAL when a head is confirmed but
the causal criteria fail, and NEGATIVE otherwise.

The criteria live in `src/experiments/content_verdict.py` — the same module
the test suite checks — so the verdict cannot drift from the measurement
and cannot crash on a metric that lives in the other manifest. The
mechanism report printed next to the verdict (the aggregate content mass,
the group ablation, the attribution) is **secondary**: it describes the
circuit, it does not decide the verdict.


In [ ]:
from pathlib import Path

from src.experiments.content_verdict import (
    format_report,
    load_manifest,
    per_seed_metrics,
    run_verdict,
)


def report(exp1_path, exp4_path=None):
    """The pre-registered verdict of one run, joined across the manifests."""
    exp1 = load_manifest(exp1_path)
    exp4 = None
    if exp4_path and Path(exp4_path).exists():
        exp4 = load_manifest(exp4_path)
    result = run_verdict(exp1, exp4)
    print(f"\n=== {exp1_path} ===")
    print(format_report(result, per_seed_metrics(exp1)))
    return result


emergence = report("results/induction_content_emergence.json")
full = report("results/induction_content.json", "results/exp4_content.json")


## After downloading

1. Unzip into a scratch dir, then pin: `python scripts/pin_remote_run.py --from <dir>` — it validates provenance (git sha must exist locally, clean tree, seeds/aggregate contract) and installs only what passes.
2. `make verify-claims` must stay green — the new manifests need their `<!-- manifest: results/... -->` tags next to the numbers they back.
3. Report exactly what happened, including "no content-based head at the full budget": the answer to issue #200 is data, either way. The pre-registration says a negative is a negative.
4. Update the Rung 1 section of `portfolio/RESULTS.md` with the new numbers, per seed, and with the verdict the criteria produced.
5. If the verdict is POSITIVE, the Rung 4 and Rung 5 sections have to say which *kind* of head they were measured on: the cascade they ran on 2026-10-20 was measured on a prefix-copying head, and a content head is a different circuit.
6. If the verdict is NEGATIVE but the mechanism report says the copy is distributed, that is a *finding* and not a null result: report it as "the copy is content-based and carried by a group of heads, not by one" — and change the headline from "a head" to "a circuit".
